In [ ]:
import os
from PIL import Image
import torch

from diffusers import QwenImageEditPipeline

pipeline = QwenImageEditPipeline.from_pretrained(
    "Qwen/Qwen-Image-Edit",
    use_safetensors=True,
    load_in_8bit=True  # Simple 8-bit loading
)
print("pipeline loaded")
pipeline.to(torch.bfloat16)
pipeline.enable_model_cpu_offload()
pipeline.set_progress_bar_config(disable=None)

In [ ]:
image = Image.open("/mnt/damian/gamescom/TRELLIS/IMG_8934.jpg").convert("RGB")
prompt = "Transform the figure into a 3D model optimized for 3D printing it as a table top figure. Make the owl a bit thinner and let it sit on his shoulder. "
inputs = {
    "image": image,
    "prompt": prompt,
    "generator": torch.manual_seed(0),
    "true_cfg_scale": 4.0,
    "negative_prompt": " ",
    "num_inference_steps": 50,
}

with torch.inference_mode():
    output = pipeline(**inputs)

output_image = output.images[0]
output_image

In [ ]:
import os
from pathlib import Path
os.environ['ATTN_BACKEND'] = 'flash-attn'   # Can be 'flash-attn' or 'xformers', default is 'flash-attn'
os.environ['SPCONV_ALGO'] = 'native'        # Can be 'native' or 'auto', default is 'auto'.
                                            # 'auto' is faster but will do benchmarking at the beginning.
                                            # Recommended to set to 'native' if run only once.
os.environ['OMP_NUM_THREADS'] = '4'        # Set the number of threads for OpenMP. Default is 4.
os.environ['TOKENIZERS_PARALLELISM'] = 'false' 

import imageio
import gc
import torch
from PIL import Image
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import postprocessing_utils

# Load a pipeline from a model folder or a Hugging Face model hub.
trellis_pipeline = TrellisImageTo3DPipeline.from_pretrained("JeffreyXiang/TRELLIS-image-large")
# pipeline = TrellisImageTo3DPipeline.from_pretrained("./assets/pretrained_TRELLIS_image_large_finetuned/snapshots/25e0d31ffbebe4b5a97464dd851910efc3002d96")
trellis_pipeline.cuda()
output_folder = Path("./")

output_image = Image.open("/mnt/damian/gamescom/TRELLIS/output_image_edit.png")

# Run the pipeline
outputs = trellis_pipeline.run(
    output_image,
    seed=13,
    sparse_structure_sampler_params={
        "steps": 28,
        "cfg_strength": 7.5,
    },
    slat_sampler_params={
        "steps": 28,
        "cfg_strength": 3.5,
    },
)

# GLB files can be extracted from the outputs
glb_file_name = "generated_3d_model.glb"
glb_file_path = output_folder / glb_file_name
# Save the GLB file
glb = postprocessing_utils.to_glb_new(
    outputs['gaussian'][0],
    outputs['mesh'][0],
    # Existing parameters
    fill_holes=True,
    fill_holes_max_size=0.4,
    simplify=0.98,
    texture_size=1024,
    brightness=1.5,
    # New parameters for floating component removal
    remove_floating=True,
    min_component_ratio=0.03,  # Remove components smaller than 3% of total faces
    min_component_faces=50,    # Or smaller than 50 faces absolute
    use_voxel_cleanup=False,   # Set to True for very aggressive cleanup
    voxel_resolution=256,
)
glb.export(glb_file_path)